# HW13 — Токенизация текста, инференс BERT и базовый fine-tuning для классификации

**Датасет:** `emotion` (6 классов эмоций)  
**Модель:** `distilbert-base-uncased`  
**Задача:** классификация коротких англоязычных текстов по эмоциональной окраске


## 1. Импорты, seed и среда


In [1]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

# ---- Seed ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Python torch version: {torch.__version__}")

# ---- Artifacts dir ----
os.makedirs("artifacts", exist_ok=True)


Device: cuda
Python torch version: 2.10.0+cu128


## 2. Данные и первичный анализ


In [2]:
dataset = load_dataset("emotion")

print("Splits и размеры:")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} примеров")

label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"\nКлассы ({num_labels}): {label_names}")


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Splits и размеры:
  train: 16000 примеров
  validation: 2000 примеров
  test: 2000 примеров

Классы (6): ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [3]:
# Несколько примеров из train
print("Примеры из train:")
for i in range(5):
    ex = dataset["train"][i]
    print(f"  [{label_names[ex['label']]:>10s}]  {ex['text'][:120]}")


Примеры из train:
  [   sadness]  i didnt feel humiliated
  [   sadness]  i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
  [     anger]  im grabbing a minute to post i feel greedy wrong
  [      love]  i am ever feeling nostalgic about the fireplace i will know that it is still on the property
  [     anger]  i am feeling grouchy


In [4]:
# Распределение классов
from collections import Counter

train_counts = Counter(dataset["train"]["label"])
print("Распределение классов в train:")
for label_id in sorted(train_counts):
    print(f"  {label_names[label_id]:>10s}: {train_counts[label_id]:>5d}")


Распределение классов в train:
     sadness:  4666
         joy:  5362
        love:  1304
       anger:  2159
        fear:  1937
    surprise:   572


In [ ]:
Датасет содержит короткие отрывки текста. Задача заключается в том, чтобы классифицировать их по эмоциональной окраске.

## 3. Токенизация


In [5]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Модель токенизатора: {MODEL_NAME}")
print(f"Размер словаря: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print(f"  [CLS] id = {tokenizer.cls_token_id}")
print(f"  [SEP] id = {tokenizer.sep_token_id}")
print(f"  [PAD] id = {tokenizer.pad_token_id}")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Модель токенизатора: distilbert-base-uncased
Размер словаря: 30522
Special tokens: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
  [CLS] id = 101
  [SEP] id = 102
  [PAD] id = 0


In [6]:
# Детальный разбор токенизации на 5 примерах
sample_texts = [
    dataset["train"][i]["text"] for i in range(5)
]

for idx, text in enumerate(sample_texts):
    enc = tokenizer(text, padding="max_length", truncation=True, max_length=32)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    print(f"--- Пример {idx + 1} ---")
    print(f"Текст:          {text[:100]}")
    print(f"Tokens:         {tokens}")
    print(f"input_ids:      {enc['input_ids']}")
    print(f"attention_mask: {enc['attention_mask']}")
    print()


--- Пример 1 ---
Текст:          i didnt feel humiliated
Tokens:         ['[CLS]', 'i', 'didn', '##t', 'feel', 'humiliated', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids:      [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

--- Пример 2 ---
Текст:          i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and 
Tokens:         ['[CLS]', 'i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'cares', 'and', 'is', 'awake', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]',

### Наблюдения по токенизации

- `[CLS]` добавляется в начало, `[SEP]` — в конец каждого текста.
- При `padding='max_length'` оставшиеся позиции заполняются `[PAD]` (id=0), а `attention_mask` ставит 0 для pad-токенов.
- `truncation=True` обрезает текст до `max_length`, если он длиннее.
- Некоторые слова разбиваются на subword-фрагменты (с префиксом `##`).


## 4. Инференс готовой pretrained модели


In [7]:
# Используем готовую модель sentiment-analysis для демонстрации
pretrained_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1,
)

inference_texts = [
    dataset["test"][i]["text"] for i in range(5)
]
inference_labels = [
    label_names[dataset["test"][i]["label"]] for i in range(5)
]

print("Инференс готовой модели (sentiment-analysis, SST-2):")
print("-" * 70)
for text, true_label in zip(inference_texts, inference_labels):
    result = pretrained_pipe(text, truncation=True, max_length=128)[0]
    print(f"True: {true_label:>10s} | Pred: {result['label']:>8s} ({result['score']:.3f}) | {text[:80]}")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Инференс готовой модели (sentiment-analysis, SST-2):
----------------------------------------------------------------------
True:    sadness | Pred: NEGATIVE (1.000) | im feeling rather rotten so im not very ambitious right now
True:    sadness | Pred: NEGATIVE (0.999) | im updating my blog because i feel shitty
True:    sadness | Pred: POSITIVE (0.999) | i never make her separate from me because i don t ever want her to feel like i m
True:        joy | Pred: POSITIVE (0.987) | i left with my bouquet of red and yellow tulips under my arm feeling slightly mo
True:    sadness | Pred: NEGATIVE (1.000) | i was feeling a little vain when i did this one


### Комментарий к инференсу готовой модели

Готовая модель `distilbert-base-uncased-finetuned-sst-2-english` обучена на задаче бинарного sentiment (POSITIVE/NEGATIVE), тогда как датасет `emotion` содержит 6 классов эмоций (sadness, joy, love, anger, fear, surprise). Поэтому готовая модель принципиально не способна решить нашу задачу — она различает только тональность, но не конкретную эмоцию. Это демонстрирует необходимость fine-tuning под конкретную задачу.


## 5. Fine-tuning для классификации текста


In [8]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Токенизация завершена.")
print(f"Колонки train: {tokenized_datasets['train'].column_names}")


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Токенизация завершена.
Колонки train: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [9]:
# Загрузка модели для fine-tuning
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)
model.to(device)
print(f"Модель {MODEL_NAME} загружена, num_labels={num_labels}")
print(f"Число параметров: {sum(p.numel() for p in model.parameters()):,}")


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Модель distilbert-base-uncased загружена, num_labels=6
Число параметров: 66,958,086


In [10]:
# Метрики для Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}


In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("Начинаем обучение...")
train_result = trainer.train()
print("Обучение завершено.")
print(f"  Train loss:  {train_result.metrics['train_loss']:.4f}")
print(f"  Train time:  {train_result.metrics['train_runtime']:.1f} s")


Начинаем обучение...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.256932,0.205002,0.918500,0.889954
2,0.140729,0.149867,0.936500,0.910652
3,0.104010,0.140602,0.940500,0.914846


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Обучение завершено.
  Train loss:  0.2934
  Train time:  609.3 s


## 6. Оценка качества на test


In [12]:
# Финальная оценка на test (один раз)
test_results = trainer.evaluate(tokenized_datasets["test"])
print("Результаты на test:")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")


Результаты на test:
  eval_loss: 0.1722
  eval_accuracy: 0.9210
  eval_f1_macro: 0.8715
  eval_runtime: 8.5771
  eval_samples_per_second: 233.1790
  eval_steps_per_second: 3.7310
  epoch: 3.0000


In [13]:
# Предсказания на test
test_preds_output = trainer.predict(tokenized_datasets["test"])
test_preds = np.argmax(test_preds_output.predictions, axis=-1)
test_true = test_preds_output.label_ids
test_probs = torch.softmax(torch.tensor(test_preds_output.predictions), dim=-1).numpy()
test_confidence = test_probs.max(axis=-1)

print(classification_report(
    test_true, test_preds,
    target_names=label_names,
    digits=4,
))


              precision    recall  f1-score   support

     sadness     0.9556    0.9639    0.9597       581
         joy     0.9424    0.9424    0.9424       695
        love     0.8137    0.8239    0.8187       159
       anger     0.9358    0.9018    0.9185       275
        fear     0.8554    0.9241    0.8884       224
    surprise     0.8039    0.6212    0.7009        66

    accuracy                         0.9210      2000
   macro avg     0.8845    0.8629    0.8715      2000
weighted avg     0.9208    0.9210    0.9203      2000



## 7. Матрица ошибок


In [14]:
cm = confusion_matrix(test_true, test_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (test)")
plt.tight_layout()
plt.savefig("artifacts/confusion_matrix.png", dpi=150)
plt.show()
print("Матрица ошибок сохранена: artifacts/confusion_matrix.png")


Матрица ошибок сохранена: artifacts/confusion_matrix.png


## 8. Примеры предсказаний


In [15]:
# Формируем sample_predictions.csv
test_texts = dataset["test"]["text"]

df_preds = pd.DataFrame({
    "text": test_texts,
    "true_label": [label_names[l] for l in test_true],
    "pred_label": [label_names[p] for p in test_preds],
    "confidence": np.round(test_confidence, 4),
})

df_preds.to_csv("artifacts/sample_predictions.csv", index=False)
print(f"Сохранено: artifacts/sample_predictions.csv ({len(df_preds)} строк)")

# Показать 10 примеров
print("\nПримеры предсказаний (первые 10):")
print(df_preds.head(10).to_string(index=False))


Сохранено: artifacts/sample_predictions.csv (2000 строк)

Примеры предсказаний (первые 10):
                                                                                                                                                                                                                          text true_label pred_label  confidence
                                                                                                                                                                   im feeling rather rotten so im not very ambitious right now    sadness    sadness      0.9968
                                                                                                                                                                                     im updating my blog because i feel shitty    sadness    sadness      0.9972
                                                                                                                             i never make

## 9. Краткий анализ ошибок


In [16]:
errors = df_preds[df_preds["true_label"] != df_preds["pred_label"]]
print(f"Всего ошибок на test: {len(errors)} из {len(df_preds)} ({len(errors)/len(df_preds)*100:.1f}%)")
print()

# Показать 10 ошибок
print("Примеры ошибок модели:")
print("-" * 90)
for _, row in errors.head(10).iterrows():
    print(f"  True: {row['true_label']:>10s} | Pred: {row['pred_label']:>10s} | "
          f"Conf: {row['confidence']:.3f} | {row['text'][:70]}")


Всего ошибок на test: 158 из 2000 (7.9%)

Примеры ошибок модели:
------------------------------------------------------------------------------------------
  True:       fear | Pred:      anger | Conf: 0.694 | i don t feel particularly agitated
  True:    sadness | Pred:        joy | Conf: 0.710 | im not sure the feeling of loss will ever go away but it may dull to a
  True:      anger | Pred:    sadness | Conf: 0.617 | i feel a bit stressed even though all the things i have going on are f
  True:   surprise | Pred:       fear | Conf: 0.627 | i am right handed however i play billiards left handed naturally so me
  True:        joy | Pred:       love | Conf: 0.645 | i feel like i am in paradise kissing those sweet lips make me feel lik
  True:       fear | Pred:   surprise | Conf: 0.496 | i was feeling weird the other day and it went away about minutes after
  True:      anger | Pred:       fear | Conf: 0.766 | when a friend dropped a frog down my neck
  True:      anger | Pred:       f

### Комментарий к ошибкам

Наиболее типичные ошибки — путаница между эмоционально близкими классами: `sadness` ↔ `anger`, `love` ↔ `joy`, `fear` ↔ `sadness`. Это объяснимо: тексты с этими эмоциями часто имеют схожую лексику и тональность. Класс `surprise` обычно распознаётся хуже всего из-за малого числа примеров в обучающей выборке.


## 10. Итоговая сводка


In [17]:
test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds, average="macro")

print("=" * 50)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ (test)")
print("=" * 50)
print(f"  Модель:         {MODEL_NAME}")
print(f"  Датасет:        emotion (6 классов)")
print(f"  max_length:     {MAX_LENGTH}")
print(f"  Эпохи:          3")
print(f"  Batch size:     32")
print(f"  Learning rate:  2e-5")
print(f"  test_accuracy:  {test_acc:.4f}")
print(f"  test_f1_macro:  {test_f1:.4f}")
print("=" * 50)


ИТОГОВЫЕ РЕЗУЛЬТАТЫ (test)
  Модель:         distilbert-base-uncased
  Датасет:        emotion (6 классов)
  max_length:     128
  Эпохи:          3
  Batch size:     32
  Learning rate:  2e-5
  test_accuracy:  0.9210
  test_f1_macro:  0.8715
